# INTC1204 Contest: Predicting Hong Kong Rental Prices

## Starter Notebook

This notebook provides a **baseline model** for the contest. It trains a simple Random Forest on basic features and generates a prediction CSV for submission.

**Your goal:** Improve upon this baseline by:
- Adding spatial features (distance to MTR, parks, CBD, etc.)
- Trying different models (Gradient Boosting, tuned trees, etc.)
- Engineering creative features from the provided datasets or external data

See the tutorial notebook for examples of spatial feature engineering!

In [ ]:
# ── Imports ──
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score

RANDOM_SEED = 42

## 1. Load Data

In [ ]:
# ── Load contest training data ──
train = pd.read_csv('./data/HK_house_transactions.csv')

print(f'Training set: {train.shape}')
print(f'Target (price) stats:')
print(train['price'].describe())
train.head()

## 2. Build Baseline Model

We use a simple Random Forest with just `area_sqft`, `floor`, and `district`.

In [ ]:
# ── Separate features and target ──
X = train.drop(columns=['price'])
y = train['price'].values

# ── Encode district ──
le = LabelEncoder()
train['district_code'] = le.fit_transform(train['district'])

# ── Baseline features ──
feature_cols = ['area_sqft', 'floor', 'district_code']
X_train = train[feature_cols].values

## 3. Evaluate with Cross-Validation

In [ ]:
# ── 5-fold cross-validation ──
model = RandomForestRegressor(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1)

cv_rmse = cross_val_score(model, X_train, y, cv=5, scoring='neg_root_mean_squared_error')
cv_r2 = cross_val_score(model, X_train, y, cv=5, scoring='r2')

print(f'5-fold CV Results:')
print(f'  RMSE: {-cv_rmse.mean():,.0f} +/- {cv_rmse.std():,.0f} HKD')
print(f'  R²:   {cv_r2.mean():.4f} +/- {cv_r2.std():.4f}')

## 4. Train Final Model and Generate Predictions

In [ ]:
# ── Train on ALL training data ──
model.fit(X_train, y)

# ── Load test features ──
test = pd.read_csv('./data/test_features.csv')
print(f'Test set: {test.shape}')

# ── Encode district for test set (use same encoder) ──
test['district_code'] = le.transform(test['district'])
X_test = test[feature_cols].values

# ── Predict ──
predictions = model.predict(X_test)

# ── Write submission CSV ──
submission = pd.DataFrame({
    'id': test['id'],
    'price': predictions.astype(int),
})
submission.to_csv('my_submission.csv', index=False)

print(f'\nSubmission saved to my_submission.csv')
print(f'  Rows: {len(submission)}')
print(f'  Predicted price range: ${predictions.min():,.0f} – ${predictions.max():,.0f}')
submission.head()

## 5. Submit to Dashboard

Upload your `my_submission.csv` to the contest dashboard:

1. Open **http://113.98.54.125:8500** in your browser
2. Enter password: `ischoolisbest`
3. Enter your name / student ID
4. Upload `my_submission.csv`
5. See your RMSE score and leaderboard ranking!

**To improve your score:**
- Add spatial features (see the tutorial notebook)
- Try different models (Gradient Boosting, tuned hyperparameters)
- Use cross-validation to test features before submitting

Good luck!